# VitroVision — Training on Kaggle GPU
EfficientNet-B0 | healthy / contaminated / dead | wandb logging

**วิธีใช้ (ภาพจริง):**
1. Upload notebook นี้ขึ้น Kaggle
2. เปิด GPU: Settings → Accelerator → GPU T4 x2
3. เพิ่ม Google Drive เป็น Dataset (Add Data → Google Drive)
4. ใส่ WANDB_API_KEY ใน Kaggle Secrets (Add-ons → Secrets)
5. Run All

**Demo mode (ยังไม่มีภาพจริง):** ตั้ง `DEMO_MODE = True` ใน cell Config — จะสร้างภาพทดสอบให้อัตโนมัติ

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'wandb', 'timm', 'albumentations', '--quiet'], check=True)
print('packages ready')

In [ ]:
import os, json, random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, f1_score
import wandb
import matplotlib.pyplot as plt
import seaborn as sns

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# ── Config ────────────────────────────────────────────────────────
DEMO_MODE = True   # True = สร้างภาพทดสอบ | False = ใช้ภาพจริงจาก Drive

DRIVE_DATA_DIR = Path('/kaggle/input/vitrovision-images')
MODEL_OUT      = Path('/kaggle/working/classifier.pt')
CKPT_DIR       = Path('/kaggle/working/checkpoints')
CKPT_DIR.mkdir(exist_ok=True)

LABELS   = ['healthy', 'contaminated', 'dead']
L2I      = {l: i for i, l in enumerate(LABELS)}
IMG_SIZE = 224

CFG = dict(
    epochs_head = 3  if DEMO_MODE else 15,
    epochs_full = 5  if DEMO_MODE else 25,
    lr          = 1e-4,
    batch_size  = 32,
)
print('Demo mode:', DEMO_MODE)
print('Config:', CFG)

In [ ]:
# ── Demo: สร้างภาพทดสอบ (รันเฉพาะตอน DEMO_MODE = True) ───────────
if DEMO_MODE:
    DEMO_DIR = Path('/kaggle/working/demo_images')
    N_PER_CLASS = 30  # 30 ภาพต่อ class = 90 ภาพรวม

    # สีประจำแต่ละ class (BGR)
    CLASS_COLORS = {
        'healthy':      (34,  139, 34),   # เขียว
        'contaminated': (255, 255,  0),   # เหลือง
        'dead':         (50,   50, 50),   # เทาเข้ม
    }

    for cls, color in CLASS_COLORS.items():
        folder = DEMO_DIR / cls
        folder.mkdir(parents=True, exist_ok=True)
        for i in range(N_PER_CLASS):
            img = np.ones((256, 256, 3), dtype=np.uint8) * np.array(color, dtype=np.uint8)
            # เพิ่ม noise เพื่อไม่ให้ภาพซ้ำกันทุกใบ
            noise = np.random.randint(-30, 30, img.shape, dtype=np.int16)
            img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
            fname = f'S01-A-{i+1:02d}_day001_{i+1:04d}_{cls}.jpg'
            cv2.imwrite(str(folder / fname), img)

    DRIVE_DATA_DIR = DEMO_DIR
    total = N_PER_CLASS * len(LABELS)
    print(f'สร้างภาพทดสอบ {total} ใบ ({N_PER_CLASS} ต่อ class) ที่ {DEMO_DIR}')
else:
    print(f'ใช้ภาพจริงจาก {DRIVE_DATA_DIR}')

In [ ]:
# ── wandb login ───────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('wandb login OK')
except Exception as e:
    print(f'wandb secret ไม่พบ ({e}) — ดำเนินการต่อโดยไม่มี wandb')
    wandb.login(anonymous='allow')

In [ ]:
# ── โหลด paths + labels จากชื่อไฟล์ ──────────────────────────────
def load_dataset(data_dir: Path):
    paths, labels = [], []
    for img_path in data_dir.rglob('*.jpg'):
        status = img_path.stem.rsplit('_', 1)[-1].lower()
        if status in LABELS:
            paths.append(str(img_path))
            labels.append(status)
    return paths, labels

paths, labels = load_dataset(DRIVE_DATA_DIR)
print(f'ภาพทั้งหมด: {len(paths)}')
print('per class:', dict(Counter(labels)))

In [ ]:
# ── Train/Val/Test Split ──────────────────────────────────────────
min_c    = min(Counter(labels).values())
use_test = len(paths) >= 30

if use_test:
    tr_p, tmp_p, tr_l, tmp_l = train_test_split(
        paths, labels, test_size=0.30,
        stratify=labels if min_c >= 3 else None, random_state=42)
    vl_p, te_p, vl_l, te_l = train_test_split(
        tmp_p, tmp_l, test_size=0.50,
        stratify=tmp_l if min(Counter(tmp_l).values()) >= 2 else None, random_state=42)
    print(f'Split 70/15/15 — Train {len(tr_p)} | Val {len(vl_p)} | Test {len(te_p)}')
else:
    tr_p, vl_p, tr_l, vl_l = train_test_split(
        paths, labels, test_size=0.2,
        stratify=labels if min_c >= 3 else None, random_state=42)
    te_p, te_l = vl_p, vl_l
    print(f'Split 80/20 — Train {len(tr_p)} | Val {len(vl_p)}')

In [ ]:
# ── Class Weights ─────────────────────────────────────────────────
tr_counter    = Counter(tr_l)
n_train       = len(tr_l)
class_weights = torch.tensor([
    n_train / (len(LABELS) * max(tr_counter.get(cls, 1), 1))
    for cls in LABELS
], dtype=torch.float).to(DEVICE)

for i, cls in enumerate(LABELS):
    print(f'  {cls}: {class_weights[i]:.3f}')

In [ ]:
# ── Dataset + Augmentation ────────────────────────────────────────
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

tr_aug = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.65, 1.0)),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3), A.RandomRotate90(p=0.5),
    A.ColorJitter(0.35, 0.35, 0.3, 0.08, p=0.8),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=mean, std=std), ToTensorV2(),
])
vl_aug = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
                    A.Normalize(mean=mean, std=std), ToTensorV2()])

class _DS(Dataset):
    def __init__(self, paths, labels, aug):
        self.paths  = paths
        self.labels = [L2I[l] for l in labels]
        self.aug    = aug
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.paths[idx]), cv2.COLOR_BGR2RGB)
        return self.aug(image=img)['image'], self.labels[idx]

tr_loader = DataLoader(_DS(tr_p, tr_l, tr_aug), batch_size=CFG['batch_size'], shuffle=True,  num_workers=2)
vl_loader = DataLoader(_DS(vl_p, vl_l, vl_aug), batch_size=CFG['batch_size'], shuffle=False, num_workers=2)
te_loader = DataLoader(_DS(te_p, te_l, vl_aug), batch_size=CFG['batch_size'], shuffle=False, num_workers=2)
print('DataLoaders ready')

In [ ]:
# ── Model + wandb init ────────────────────────────────────────────
model     = timm.create_model('efficientnet_b0', pretrained=True, num_classes=3).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
best_loss = float('inf')

run = wandb.init(
    project='vitrovision',
    name=f'{"demo" if DEMO_MODE else "kaggle"}-gpu-b{CFG["batch_size"]}',
    config={**CFG, 'model': 'efficientnet_b0', 'img_size': IMG_SIZE,
            'demo_mode': DEMO_MODE,
            'n_train': len(tr_p), 'n_val': len(vl_p), 'n_test': len(te_p),
            'device': DEVICE,
            'class_weights': {cls: round(class_weights[i].item(), 3)
                              for i, cls in enumerate(LABELS)}},
    tags=['capsicum', 'tc-monitoring', 'demo' if DEMO_MODE else 'kaggle-gpu'],
)
print('wandb run:', run.url)

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────
def epoch_loop(loader, opt=None):
    model.train() if opt else model.eval()
    tot_loss = correct = n = 0
    ctx = torch.enable_grad() if opt else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if opt:
                opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(tgts)
            correct  += (out.argmax(1) == tgts).sum().item()
            n        += len(tgts)
    return tot_loss / n, correct / n

def phase(phase_num, total_ep, opt, scheduler):
    global best_loss
    print(f'\n=== Phase {phase_num} ({total_ep} epochs) ===')
    for ep in range(1, total_ep + 1):
        tl, ta = epoch_loop(tr_loader, opt)
        vl, va = epoch_loop(vl_loader)
        scheduler.step()
        lr = opt.param_groups[0]['lr']
        if vl < best_loss:
            best_loss = vl
            torch.save(model, CKPT_DIR / 'best.pt')
        print(f'Ep {ep:02d}/{total_ep} | tl={tl:.4f} ta={ta*100:.1f}% | vl={vl:.4f} va={va*100:.1f}% | lr={lr:.2e}')
        wandb.log({'train/loss': tl, 'train/acc': ta*100,
                   'val/loss': vl,   'val/acc':   va*100,
                   'lr': lr, 'phase': phase_num})

# Phase 1 — head only
for p in model.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = True
opt1 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CFG['lr'])
sch1 = CosineAnnealingLR(opt1, T_max=CFG['epochs_head'], eta_min=CFG['lr']*0.01)
phase(1, CFG['epochs_head'], opt1, sch1)

# Phase 2 — fine-tune all
for p in model.parameters(): p.requires_grad = True
opt2 = torch.optim.Adam(model.parameters(), lr=CFG['lr']*0.1)
sch2 = CosineAnnealingLR(opt2, T_max=CFG['epochs_full'], eta_min=CFG['lr']*0.001)
phase(2, CFG['epochs_full'], opt2, sch2)

print('\nTraining done!')

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────
best_model = torch.load(str(CKPT_DIR / 'best.pt'), map_location=DEVICE, weights_only=False)
best_model.eval()

preds, trues = [], []
with torch.no_grad():
    for imgs, tgts in te_loader:
        imgs = imgs.to(DEVICE)
        preds.extend(best_model(imgs).argmax(1).cpu().tolist())
        trues.extend(tgts.tolist())

kappa  = cohen_kappa_score(trues, preds) if len(set(trues)) > 1 else 0.0
wf1    = f1_score(trues, preds, average='weighted', zero_division=0)
report = classification_report(trues, preds, target_names=LABELS, output_dict=True, zero_division=0)
cm     = confusion_matrix(trues, preds, labels=list(range(3)))

print(f'\nCohen Kappa : {kappa:.4f}')
print(f'Weighted F1 : {wf1:.4f}')
print('\n' + classification_report(trues, preds, target_names=LABELS, zero_division=0))

In [ ]:
# ── Confusion Matrix + wandb ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix | κ={kappa:.3f} | F1={wf1:.3f}')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

wandb.log({
    'test/kappa':       kappa,
    'test/weighted_f1': wf1,
    'test/confusion_matrix': wandb.plot.confusion_matrix(
        probs=None, y_true=trues, preds=preds, class_names=LABELS),
    'confusion_matrix_img': wandb.Image('/kaggle/working/confusion_matrix.png'),
    **{f'test/{cls}/recall':    report[cls]['recall']    for cls in LABELS},
    **{f'test/{cls}/f1':        report[cls]['f1-score']  for cls in LABELS},
    **{f'test/{cls}/precision': report[cls]['precision'] for cls in LABELS},
})
wandb.finish()
print('wandb finished')

In [ ]:
# ── บันทึก Model + Metrics ────────────────────────────────────────
torch.save(best_model, MODEL_OUT)
print(f'Model saved: {MODEL_OUT}')

with open('/kaggle/working/metrics.json', 'w') as f:
    json.dump({
        'cohen_kappa': round(kappa, 4), 'weighted_f1': round(wf1, 4),
        'confusion_matrix': cm.tolist(), 'labels': LABELS,
        'classification_report': {k: v for k, v in report.items() if k != 'accuracy'},
        'class_weights': {cls: round(class_weights[i].item(), 3) for i, cls in enumerate(LABELS)},
        'n_train': len(tr_p), 'n_val': len(vl_p), 'n_test': len(te_p),
        'device': DEVICE, 'demo_mode': DEMO_MODE,
    }, f, indent=2)

print('\nDownload จาก Output tab:')
print('  classifier.pt       — โมเดล')
print('  metrics.json        — ผลการประเมิน')
print('  confusion_matrix.png')